# Laboratorio 1 · Entorno y primeros casos comprobados

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/udd-ia-generativa-2026/curso-ia/blob/main/notebooks/lab01.ipynb)

**Formativo, sin nota.** Trabajo individual en Google Colab: instalas el código del curso, ejecutas un caso común en dos modelos y entregas tu evidencia en Canvas.

Al terminar tendrás: una llamada a un modelo escrita por ti, casos ejecutados en dos modelos, una revisión de cada respuesta y una tabla con tokens, latencia y costo.

Antes de empezar, abre **Secretos** (el ícono de llave del panel izquierdo) y agrega `CURSO_API_KEY` y tu clave de Gemini como `GOOGLE_API_KEY` o `GEMINI_API_KEY` (el nombre que usa el botón de Colab), con acceso para este notebook. No escribas claves en las celdas ni imprimas `os.environ`.

Al instalar, pip puede avisar un conflicto de `google-auth` con `google-colab`: es esperado y no afecta al laboratorio. Si el docente publica una versión nueva durante la clase, usa **Entorno de ejecución → Reiniciar sesión** y vuelve a ejecutar desde la primera celda.

In [ ]:
import os
import sys

# En Colab se instala el código del curso; en un repositorio con entorno propio ya está instalado.
if "COLAB_RELEASE_TAG" in os.environ:
    %pip install -q --upgrade "curso-ia @ git+https://github.com/udd-ia-generativa-2026/curso-ia@main"
    if "curso_ia" in sys.modules:
        print("El paquete ya estaba cargado: usa Entorno de ejecución → Reiniciar sesión y vuelve a ejecutar desde aquí.")


In [ ]:
import pandas as pd
from pydantic_ai import Agent

from curso_ia import (
    blocked, create_model, load_cases, load_config, merge_review, prepare_lab, run_case, save_run, status,
)

config = load_config()
lab = prepare_lab(1)  # copia los casos a la carpeta lab01 sin borrar lo que ya editaste
if config.mode == "simulated":
    print("MODO SIMULADO: respuestas simuladas; no acredita acceso a ningún modelo.")
pd.DataFrame(status(config))


La tabla muestra qué modelo atiende cada rol (`proveedor:modelo`) y, si falta algo, el **nombre** de la variable; nunca su valor. `generation` usa el servidor del curso y `comparison` usa Gemini; con una clave propia puedes cambiar un rol sin tocar el código (variables `GENERATION_MODEL` y `COMPARISON_MODEL`).

## 1. Tu primera llamada

Un `Agent` de PydanticAI recibe un modelo y una instrucción; `run` envía la consulta y devuelve la respuesta (`output`) y el consumo (`usage`), con los tokens de entrada y de salida que determinan el costo. La celda usa el primer perfil que esté listo.

In [ ]:
ready = [row["role"] for row in status(config) if row["ready"]]
role = ready[0] if ready else "generation"  # el primer perfil listo; si ninguno, muestra qué falta
print("Perfil:", role, config.models[role])
model = create_model(config.models[role], config)
agent = Agent(model, instructions="Responde en una sola frase.")
result = await agent.run("¿Qué es un token en un modelo de lenguaje?")
print(result.output)
print(result.usage)


## 2. Los casos

`lab01/cases.toml` trae el caso común del curso: la regla de tarifas por duelo del caso Air Canada en cuatro consultas, cada una con su respuesta esperada. Ejecútalo sin cambios.

Mínimo para hoy: el caso común en ambos perfiles. Extensión: abre el panel **Archivos**, edita `lab01/my-task.toml` con dos o tres casos de la tarea que trajiste del material previo, cambia `CASES_FILE` y vuelve a ejecutar desde aquí.

In [ ]:
CASES_FILE = "cases.toml"  # extensión: "my-task.toml"
cases = load_cases(lab / CASES_FILE)
if cases.is_example:
    print("Aún usas los casos de ejemplo: reemplázalos por los de tu tarea.")
print("Instrucción:", cases.instruction)
pd.DataFrame([vars(case) for case in cases.cases])


## 3. Ejecutar en ambos perfiles

Cada caso se envía por separado, sin mensajes anteriores. Si un perfil no está configurado o falla, se registra el bloqueo y se sigue con el otro.

Extensión: sube `REPETITIONS` a 2 o 3 para ver cuánto varían la respuesta, los tokens y la latencia.

In [ ]:
REPETITIONS = 1
SETTINGS = {}  # por ejemplo {"temperature": 0.2}; vacío usa el valor de cada proveedor

rows = []
for profile, ref in config.models.items():
    try:
        agent = Agent(create_model(ref, config), instructions=cases.instruction, model_settings=SETTINGS)
    except Exception as error:
        rows += [blocked(profile, ref, case, config, 1, error) for case in cases.cases]
        continue
    for case in cases.cases:
        for repetition in range(1, REPETITIONS + 1):
            rows.append(await run_case(agent, case, profile=profile, ref=ref, config=config, repetition=repetition))

columns = ["profile", "case", "repetition", "status", "response", "input_tokens", "output_tokens", "latency_ms", "estimated_cost_usd"]
pd.DataFrame(rows).reindex(columns=columns)


## 4. Guardar la ejecución

Se guardan dos archivos en `lab01/evidence/`: la ejecución completa en JSON, que no depende de las salidas del notebook, y un archivo de revisión para completar a mano.

In [ ]:
json_path, review_path = save_run(rows, cases, config, lab / "evidence", lab=1, settings=SETTINGS)
print("Ejecución:", json_path.relative_to(lab.parent))
print("Revisión: ", review_path.relative_to(lab.parent))


## 5. Revisar cada respuesta

Abre el archivo de revisión desde el panel **Archivos** (carpeta `lab01/evidence`). Para cada respuesta compara con la esperada y completa `passed` con `true` o `false` (sin comillas) y `reason` con una frase: qué coincide, qué falta o qué se agregó sin respaldo. No edites la respuesta: el JSON conserva la original. Una respuesta incorrecta es tan útil como una correcta si explicas por qué.

Guarda el archivo y ejecuta la celda siguiente; puedes repetirla las veces que quieras.

In [ ]:
evidence = merge_review(json_path, review_path, config)
print("Revisión completa:", evidence["review_complete"])

numeric = ["input_tokens", "output_tokens", "latency_ms", "estimated_cost_usd"]
reviewed = pd.DataFrame(evidence["results"]).reindex(columns=["profile", "status", "passed", *numeric])
reviewed[numeric] = reviewed[numeric].apply(pd.to_numeric, errors="coerce")
reviewed.groupby("profile").agg(
    responses=("status", lambda s: (s == "ok").sum()),
    passed=("passed", lambda s: (s == True).sum()),  # noqa: E712
    input_tokens=("input_tokens", "mean"),
    output_tokens=("output_tokens", "mean"),
    latency_ms=("latency_ms", "median"),
    cost_usd=("estimated_cost_usd", lambda s: s.sum(min_count=1)),
)


Lee la tabla con cuidado: con pocos casos no se puede concluir qué modelo es mejor; es una primera medición. El costo es una estimación: la fuente de cada precio queda en el JSON, y en el nivel gratuito de Gemini no pagas el precio de lista que se muestra.

## 6. Entregar y explicar

La celda siguiente comprime `lab01` con tus casos, la ejecución y la revisión, y lo descarga. Descarga también este notebook con sus salidas (**Archivo → Descargar → Descargar .ipynb**). Los archivos de Colab se borran al cerrar la sesión: descarga antes de salir.

En la tarea del laboratorio 1 en Canvas, sube ambos archivos y explica en pocas líneas un caso que falló o un costo que te sorprendió, y qué cambiarías. Si usaste un asistente, declara su uso según las reglas del curso. Si ya tienes GitHub, también puedes guardar estos archivos en un repositorio propio; no es obligatorio.

Las parejas se forman después de esta clase; tu tarea y tus casos te servirán para proponer el tema del proyecto.

In [ ]:
import shutil

archive = shutil.make_archive("lab01-submission", "zip", lab.parent, lab.name)
if "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import files
    files.download(archive)
print("Archivo para Canvas:", archive)
